In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Dual Specialist Classifier: LightGBM for ESI 2/3 & XGBoost for ESI 4/5 (`models/xgboost_esi23_esi45_extreme.ipynb`)

This notebook trains a **Dual Specialist Ensemble Model** combining **LightGBM** (for ESI 2/3 detection) and **XGBoost** (for ESI 4/5 detection) using **45 Predictor Features** (19 raw inputs from `config/triage_conf.json` + 10 binary vital anomaly flags + 16 continuous vital delta & range features), **Factor-Controlled Minority Class Random Upsampling**, 5-Fold Stratified Cross-Validation, **Balanced Accuracy**, **Specificity**, and **Matthews Correlation Coefficient (MCC)**:

### System Architecture & Workflow
1. **Specialist Model 1 (LightGBM for ESI 2/3)**:
   - Target: Binary `ESI 2/3` (`1` if ESI 2 or 3, `0` otherwise).
   - Algorithm: LightGBM (`objective = "binary"`, `metric = "binary_logloss"`).
   - Predicts $P_{\text{LGB}}(\text{ESI 2/3})$.
2. **Specialist Model 2 (XGBoost for ESI 4/5)**:
   - Target: Binary `ESI 4/5` (`1` if ESI 4 or 5, `0` otherwise).
   - Algorithm: XGBoost (`objective = "binary:logistic"`, `eval_metric = "logloss"`).
   - Predicts $P_{\text{XGB}}(\text{ESI 4/5})$.
3. **Combined Joint Probability Inference (`"2_3"`, `"4_5"`, `"other"`)**:
   - $P(\text{ESI 2/3}) = P_{\text{LGB}}(\text{ESI 2/3})$
   - $P(\text{ESI 4/5}) = P_{\text{XGB}}(\text{ESI 4/5})$
   - $P(\text{Other}) = \max(0, 1 - P(\text{ESI 2/3}) - P(\text{ESI 4/5}))$
4. **Predictor Feature Selection (45 Total Features)**:
   - **19 Raw Inputs (From `triage_conf.json`)**: `age`, `gender`, `cc_breathingdifficulty`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`, `pulse_last`, `resp_last`, `spo2_last`, `sbp_last`, `pulse_min`, `resp_min`, `spo2_min`, `sbp_min`, `pulse_max`, `resp_max`, `spo2_max`, `sbp_max`.
   - **10 Binary Vital Anomaly Flags**: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
   - **16 Continuous Vital Delta & Range Features**: `hr_mean_to_last`, `sbp_mean_to_last`, `spo2_mean_to_last`, `rr_mean_to_last`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_min`, `sbp_last_to_min`, `hr_last_to_max`, `rr_last_to_max`, `spo2_last_to_max`, `sbp_last_to_max`.
5. **Data Partitioning & Oversampling**:
   - Reserves a **15% Holdout Test Set** evaluated ONCE at the end.
   - Evaluates the **85% Validation Set** via **5-Fold Stratified Cross-Validation**.
   - Applies **Minority Class Bootstrap Random Upsampling** (`upsample_ratio = 1.0`) strictly to training partitions.
6. **Full Metrics Suite Evaluation**: Computes Per-Class and Macro-Averaged Accuracy, **Balanced Accuracy**, **Specificity**, Precision, Recall/Sensitivity, F1 Score, ROC-AUC, and **MCC Score**.
7. **Reports & Artifacts**:
   - **CSV Reports**: `reports/xgboost_esi23_esi45_5fold_cv_report.csv` and `reports/xgboost_esi23_esi45_test_report.csv`.
   - **Diagnostic Plots**: Metrics bar chart (`plots/xgboost_esi23_esi45_metrics_barchart.png`).
   - **Model Export**: Saved separately to `deploy/lightgbm_esi23_model.rds`, `deploy/xgboost_esi45_model.rds`, and `deploy/xgboost_esi23_esi45_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
})
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) {
  library(lightgbm)
  cat("LightGBM R package successfully loaded for ESI 2/3 Specialist.\n")
} else {
  cat("Note: LightGBM R package not installed. Using XGBoost binary gradient boosting fallback for ESI 2/3 Specialist.\n")
}
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 45 Features & Define Binary Target Labels
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 45 Predictor Features (19 raw triage_conf.json + 10 binary flags + 16 continuous deltas/ranges)
df_full <- data.frame(
  # 19 Raw Inputs from triage_conf.json
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  # 10 Binary Vital Anomaly Flags
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  # 16 Continuous Vital Delta & Range Features
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
# Define multi-class & specialist binary targets
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col   <- factor(ifelse(raw_esi %in% c("2", "3"), "2_3", ifelse(raw_esi %in% c("4", "5"), "4_5", "other")), levels = c("2_3", "4_5", "other"))
df_full$target_esi23 <- ifelse(raw_esi %in% c("2", "3"), 1, 0)  # LightGBM target
df_full$target_esi45 <- ifelse(raw_esi %in% c("4", "5"), 1, 0)  # XGBoost target
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA raw features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Dataset Ready: %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Grouped Class Distribution ('2_3', '4_5', 'other'):\n")
print(table(df_full$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning & 5-Fold CV (LightGBM ESI 2/3 + XGBoost ESI 4/5)
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size # 0.15
# Partition into Train/Val Set (85%) and Holdout Test Set (15%)
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
upsample_ratio <- 1.0
upsample_multiclass <- function(df_train, ratio = 1.0) {
  counts <- table(df_train$target_col)
  max_cnt <- max(counts)
  
  res_df <- df_train
  for (cls in names(counts)) {
    cls_cnt <- counts[[cls]]
    target_cnt <- round(max_cnt * ratio)
    if (target_cnt > cls_cnt) {
      extra_needed <- target_cnt - cls_cnt
      cls_subset   <- df_train[df_train$target_col == cls, ]
      sampled_extra <- cls_subset[sample(1:cls_cnt, size = extra_needed, replace = TRUE), ]
      res_df       <- rbind(res_df, sampled_extra)
    }
  }
  return(res_df)
}
cat("=== Data Partitioning Summary ===\n")
cat(sprintf("Full Dataset        : %d rows\n", nrow(df_full)))
cat(sprintf("Train/Val Set (85%%) : %d rows\n", nrow(train_val_df)))
cat(sprintf("Holdout Test Set (15%%): %d rows\n\n", nrow(test_df)))
k_folds <- 5
folds   <- createFolds(train_val_df$target_col, k = k_folds, list = TRUE, returnTrain = FALSE)
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
feature_cols <- setdiff(names(train_val_df), c(binary_cols, "target_col", "target_esi23", "target_esi45"))
cont_cols    <- feature_cols
val_fold_accs     <- numeric(k_folds)
val_fold_bal_accs <- numeric(k_folds)
val_fold_specs    <- numeric(k_folds)
cat("============================================================\n")
cat(sprintf("   STARTING %d-FOLD CV: LIGHTGBM (ESI 2/3) + XGBOOST (ESI 4/5)\n", k_folds))
cat("============================================================\n")
for (k in 1:k_folds) {
  val_idx    <- folds[[k]]
  train_fold <- train_val_df[-val_idx, ]
  val_fold   <- train_val_df[val_idx, ]
  
  train_fold <- upsample_multiclass(train_fold, ratio = upsample_ratio)
  
  preproc_fold <- preProcess(train_fold[, cont_cols, drop = FALSE], method = c("center", "scale"))
  train_fold   <- predict(preproc_fold, train_fold)
  val_fold     <- predict(preproc_fold, val_fold)
  
  all_feats <- c(binary_cols, cont_cols)
  
  train_x <- as.matrix(train_fold[, all_feats])
  val_x   <- as.matrix(val_fold[, all_feats])
  
  # 1. Train LightGBM Model for ESI 2/3
  y_tr_esi23  <- train_fold$target_esi23
  y_val_esi23 <- val_fold$target_esi23
  
  if (has_lgb) {
    dtr_lgb <- lgb.Dataset(data = train_x, label = y_tr_esi23)
    dvl_lgb <- lgb.Dataset(data = val_x,   label = y_val_esi23, reference = dtr_lgb)
    
    lgb_params <- list(
      objective = "binary", metric = "binary_logloss", learning_rate = 0.05,
      num_leaves = 31, max_depth = 6, feature_fraction = 0.8, bagging_fraction = 0.8, verbosity = -1
    )
    model_lgb_23 <- lgb.train(params = lgb_params, data = dtr_lgb, nrounds = 150, valids = list(val = dvl_lgb), early_stopping_rounds = 20, verbose = -1)
    probs_esi23  <- predict(model_lgb_23, val_x)
  } else {
    dtr_xgb23 <- xgb.DMatrix(data = train_x, label = y_tr_esi23)
    dvl_xgb23 <- xgb.DMatrix(data = val_x,   label = y_val_esi23)
    model_lgb_23 <- xgb.train(params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6), data = dtr_xgb23, nrounds = 150, watchlist = list(val = dvl_xgb23), early_stopping_rounds = 20, verbose = 0)
    probs_esi23  <- predict(model_lgb_23, dvl_xgb23)
  }
  
  # 2. Train XGBoost Model for ESI 4/5
  y_tr_esi45  <- train_fold$target_esi45
  y_val_esi45 <- val_fold$target_esi45
  
  dtr_xgb45 <- xgb.DMatrix(data = train_x, label = y_tr_esi45)
  dvl_xgb45 <- xgb.DMatrix(data = val_x,   label = y_val_esi45)
  
  model_xgb_45 <- xgb.train(
    params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6, subsample = 0.8, colsample_bytree = 0.8),
    data = dtr_xgb45, nrounds = 150, watchlist = list(val = dvl_xgb45), early_stopping_rounds = 20, verbose = 0
  )
  probs_esi45 <- predict(model_xgb_45, dvl_xgb45)
  
  # 3. Combine Predictions
  probs_other <- pmax(0, 1 - probs_esi23 - probs_esi45)
  probs_mat   <- cbind(probs_esi23, probs_esi45, probs_other)
  colnames(probs_mat) <- c("2_3", "4_5", "other")
  
  fold_pred_idx <- apply(probs_mat, 1, which.max)
  fold_pred_fac <- factor(colnames(probs_mat)[fold_pred_idx], levels = c("2_3", "4_5", "other"))
  fold_cm       <- confusionMatrix(fold_pred_fac, val_fold$target_col)
  fold_acc      <- as.numeric(fold_cm$overall["Accuracy"])
  fold_bal_acc  <- mean(fold_cm$byClass[, "Balanced Accuracy"])
  fold_spec     <- mean(fold_cm$byClass[, "Specificity"])
  
  val_fold_accs[k]     <- fold_acc
  val_fold_bal_accs[k] <- fold_bal_acc
  val_fold_specs[k]    <- fold_spec
  
  cat(sprintf("  Validation Fold %d/%d Acc: %.4f | BalAcc: %.4f | Specificity: %.4f\n",
              k, k_folds, fold_acc, fold_bal_acc, fold_spec))
}
cat("============================================================\n")
cat(sprintf("   5-FOLD CV COMPLETE. Mean Acc = %.4f | Mean BalAcc = %.4f | Mean Spec = %.4f\n",
            mean(val_fold_accs), mean(val_fold_bal_accs), mean(val_fold_specs)))
cat("============================================================\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Final Production Dual Specialist Model Training & Holdout Test Benchmark
# ---------------------------------------------------------
cat("Applying oversampling to full train_val_df (85% data)...\n")
train_val_upsampled <- upsample_multiclass(train_val_df, ratio = upsample_ratio)
all_feats <- c(binary_cols, cont_cols)
preproc_tv <- preProcess(train_val_upsampled[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_val_scaled <- predict(preproc_tv, train_val_upsampled)
test_scaled      <- predict(preproc_tv, test_df)
train_x <- as.matrix(train_val_scaled[, all_feats])
test_x  <- as.matrix(test_scaled[, all_feats])
# Train Final LightGBM for ESI 2/3
y_tr_23 <- train_val_scaled$target_esi23
y_te_23 <- test_scaled$target_esi23
if (has_lgb) {
  dtr_lgb <- lgb.Dataset(data = train_x, label = y_tr_23)
  dte_lgb <- lgb.Dataset(data = test_x,  label = y_te_23, reference = dtr_lgb)
  final_lgb_23 <- lgb.train(
    params = list(objective = "binary", metric = "binary_logloss", learning_rate = 0.05, num_leaves = 31, max_depth = 6, verbosity = -1),
    data = dtr_lgb, nrounds = 150, valids = list(test = dte_lgb), early_stopping_rounds = 20, verbose = -1
  )
  test_probs_23 <- predict(final_lgb_23, test_x)
} else {
  dtr_xgb23 <- xgb.DMatrix(data = train_x, label = y_tr_23)
  dte_xgb23 <- xgb.DMatrix(data = test_x,  label = y_te_23)
  final_lgb_23 <- xgb.train(
    params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6),
    data = dtr_xgb23, nrounds = 150, watchlist = list(test = dte_xgb23), early_stopping_rounds = 20, verbose = 0
  )
  test_probs_23 <- predict(final_lgb_23, dte_xgb23)
}
# Train Final XGBoost for ESI 4/5
y_tr_45 <- train_val_scaled$target_esi45
y_te_45 <- test_scaled$target_esi45
dtr_xgb45 <- xgb.DMatrix(data = train_x, label = y_tr_45)
dte_xgb45 <- xgb.DMatrix(data = test_x,  label = y_te_45)
final_xgb_45 <- xgb.train(
  params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6, subsample = 0.8, colsample_bytree = 0.8),
  data = dtr_xgb45, nrounds = 150, watchlist = list(test = dte_xgb45), early_stopping_rounds = 20, verbose = 0
)
test_probs_45    <- predict(final_xgb_45, dte_xgb45)
test_probs_other <- pmax(0, 1 - test_probs_23 - test_probs_45)
raw_test_probs <- cbind(test_probs_23, test_probs_45, test_probs_other)
colnames(raw_test_probs) <- c("2_3", "4_5", "other")
test_pred_idx <- apply(raw_test_probs, 1, which.max)
test_pred_fac <- factor(colnames(raw_test_probs)[test_pred_idx], levels = c("2_3", "4_5", "other"))
act_test_fac  <- factor(test_df$target_col, levels = c("2_3", "4_5", "other"))
cm_test  <- confusionMatrix(test_pred_fac, act_test_fac)
acc_test <- as.numeric(cm_test$overall["Accuracy"])
prec_by_class    <- as.numeric(cm_test$byClass[, "Pos Pred Value"])
rec_by_class     <- as.numeric(cm_test$byClass[, "Sensitivity"])
spec_by_class    <- as.numeric(cm_test$byClass[, "Specificity"])
bal_acc_by_class <- as.numeric(cm_test$byClass[, "Balanced Accuracy"])
prec_by_class[is.na(prec_by_class)]       <- 0
rec_by_class[is.na(rec_by_class)]         <- 0
spec_by_class[is.na(spec_by_class)]       <- 0
bal_acc_by_class[is.na(bal_acc_by_class)] <- 0
f1_by_class <- ifelse((prec_by_class + rec_by_class) > 0, 
                      2 * (prec_by_class * rec_by_class) / (prec_by_class + rec_by_class), 0)
roc_auc_by_class <- sapply(1:3, function(i) {
  cls_name <- levels(act_test_fac)[i]
  act_bin  <- ifelse(act_test_fac == cls_name, 1, 0)
  r_obj    <- tryCatch(pROC::roc(act_bin, raw_test_probs[, i]), error = function(e) NULL)
  if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
})
mcc_by_class <- sapply(1:3, function(i) {
  cls <- levels(act_test_fac)[i]
  tp  <- sum(test_pred_fac == cls & act_test_fac == cls)
  tn  <- sum(test_pred_fac != cls & act_test_fac != cls)
  fp  <- sum(test_pred_fac == cls & act_test_fac != cls)
  fn  <- sum(test_pred_fac != cls & act_test_fac == cls)
  
  num   <- (tp * tn) - (fp * fn)
  denom <- sqrt(as.numeric(tp + fp) * as.numeric(tp + fn) * as.numeric(tn + fp) * as.numeric(tn + fn))
  if (is.na(denom) || denom == 0) 0 else num / denom
})
actual_counts <- as.numeric(table(act_test_fac))
pred_counts   <- as.numeric(table(test_pred_fac))
diff_vec      <- pred_counts - actual_counts
diff_str      <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
test_report_df <- data.frame(
  Class             = levels(act_test_fac),
  Actual_Count      = actual_counts,
  Pred_Count        = pred_counts,
  Diff              = diff_str,
  Precision         = round(prec_by_class, 4),
  Recall_Sens       = round(rec_by_class, 4),
  Specificity       = round(spec_by_class, 4),
  Balanced_Accuracy = round(bal_acc_by_class, 4),
  F1_Score          = round(f1_by_class, 4),
  ROC_AUC           = round(roc_auc_by_class, 4),
  MCC_Score         = round(mcc_by_class, 4)
)
macro_prec    <- mean(prec_by_class)
macro_rec     <- mean(rec_by_class)
macro_spec    <- mean(spec_by_class)
macro_bal_acc <- mean(bal_acc_by_class)
macro_f1      <- mean(f1_by_class)
macro_roc_auc <- mean(roc_auc_by_class, na.rm = TRUE)
macro_mcc     <- mean(mcc_by_class)
cat(sprintf("============================================================\n"))
cat(sprintf("   DUAL SPECIALIST (LIGHTGBM ESI 2/3 + XGBOOST ESI 4/5) - TEST BENCHMARK\n"))
cat(sprintf("============================================================\n"))
cat(sprintf("  5-Fold CV Mean Val Acc : %.4f (%.2f%%)\n", mean(val_fold_accs), mean(val_fold_accs) * 100))
cat(sprintf("  5-Fold CV Mean Bal Acc : %.4f (%.2f%%)\n", mean(val_fold_bal_accs), mean(val_fold_bal_accs) * 100))
cat(sprintf("  Final Test Accuracy    : %.4f (%.2f%%)\n", acc_test, acc_test * 100))
cat(sprintf("  Macro Precision        : %.4f\n", macro_prec))
cat(sprintf("  Macro Recall (Sens)    : %.4f\n", macro_rec))
cat(sprintf("  Macro Specificity      : %.4f\n", macro_spec))
cat(sprintf("  Macro Balanced Acc     : %.4f\n", macro_bal_acc))
cat(sprintf("  Macro F1-Score         : %.4f\n", macro_f1))
cat(sprintf("  Macro ROC-AUC          : %.4f\n", macro_roc_auc))
cat(sprintf("  Macro MCC Score        : %.4f\n", macro_mcc))
cat(sprintf("============================================================\n\n"))
cat("Holdout Test Set Per-Class Metrics (with Specificity & Balanced Accuracy):\n")
print(test_report_df)
cat("\nHoldout Test Set 3x3 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
print(cm_test$table)
cat(sprintf("============================================================\n\n"))
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
cv_summary_df <- data.frame(
  Fold = paste0("Fold_", 1:k_folds),
  Validation_Accuracy = round(val_fold_accs, 4),
  Validation_Balanced_Accuracy = round(val_fold_bal_accs, 4),
  Validation_Specificity = round(val_fold_specs, 4)
)
write.csv(cv_summary_df,  file = file.path(reports_dir, "xgboost_esi23_esi45_5fold_cv_report.csv"), row.names = FALSE)
write.csv(test_report_df, file = file.path(reports_dir, "xgboost_esi23_esi45_test_report.csv"),     row.names = FALSE)
cat("5-Fold CV Validation CSV Report written to: reports/xgboost_esi23_esi45_5fold_cv_report.csv\n")
cat("Holdout Test Set CSV Report written to:       reports/xgboost_esi23_esi45_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Diagnostic Plots (Metrics Bar Chart including Specificity & MCC)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Metric = factor(c("5Fold_Val_Acc", "5Fold_Val_BalAcc", "Test_Accuracy", "Test_Macro_BalAcc", "Test_Macro_Prec", "Test_Macro_Rec", "Test_Macro_Spec", "Test_Macro_F1", "Test_Macro_AUC", "Test_Macro_MCC"),
                  levels = c("5Fold_Val_Acc", "5Fold_Val_BalAcc", "Test_Accuracy", "Test_Macro_BalAcc", "Test_Macro_Prec", "Test_Macro_Rec", "Test_Macro_Spec", "Test_Macro_F1", "Test_Macro_AUC", "Test_Macro_MCC")),
  Score  = c(mean(val_fold_accs), mean(val_fold_bal_accs), acc_test, macro_bal_acc, macro_prec, macro_rec, macro_spec, macro_f1, macro_roc_auc, macro_mcc)
)
p_bar <- ggplot(metrics_summary, aes(x = Metric, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", width = 0.5) +
  geom_text(aes(label = sprintf("%.3f", Score)), vjust = -0.3, size = 3.2, fontface = "bold") +
  theme_minimal() +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "Dual Specialist Ensemble (LightGBM ESI 2/3 + XGBoost ESI 4/5)",
       subtitle = sprintf("5-Fold CV vs Holdout Test Set Evaluation (45 Features, Upsample Ratio = %.2f)", upsample_ratio),
       y = "Metric Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "none")
ggsave(file.path(plots_dir, "xgboost_esi23_esi45_metrics_barchart.png"), plot = p_bar, width = 11, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/xgboost_esi23_esi45_metrics_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Final Production Dual Specialist Model Artifacts Separately & Combined
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
# Save LightGBM ESI 2/3 Specialist Separately
lgb_path <- file.path(deploy_dir, "lightgbm_esi23_model.rds")
saveRDS(list(model = final_lgb_23, preproc = preproc_tv, has_lgb = has_lgb), file = lgb_path)
cat("Separate LightGBM ESI 2/3 Model saved to:", lgb_path, "\n")
# Save XGBoost ESI 4/5 Specialist Separately
xgb_path <- file.path(deploy_dir, "xgboost_esi45_model.rds")
saveRDS(list(model = final_xgb_45, preproc = preproc_tv), file = xgb_path)
cat("Separate XGBoost ESI 4/5 Model saved to: ", xgb_path, "\n")
# Save Combined Dual Specialist Artifact
comb_path <- file.path(deploy_dir, "xgboost_esi23_esi45_extreme_model.rds")
saveRDS(list(model_lgb = final_lgb_23, model_xgb = final_xgb_45, preproc = preproc_tv, upsample_ratio = upsample_ratio, has_lgb = has_lgb), file = comb_path)
cat("Combined Dual Specialist Model saved to:", comb_path, "\n")